# 面试问题：LLaVA 风格多模态模型怎样把图像 token 接入 LLM？

**回答主线。** 视觉编码器把图像切成 patch token，projector 把视觉维度映射到语言 hidden size，再用 `<image>` 占位符把视觉 token 插入文本 embedding 序列。融合后的序列进入 causal decoder；图像位置通常不计算语言 token loss，assistant 文本才监督。常见两阶段训练先冻结视觉塔和 LLM 对齐 projector，再解冻部分模块做视觉指令微调。

下面只用 PyTorch 基础层手写 patch encoder、MLP projector、序列拼接、causal attention block 和 `TinyLLaVA.forward`，不导入现成 ViT、Transformer 或多模态模型。


In [ ]:
import math  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

# 小尺寸模型用于验证 shape、mask、梯度和冻结合同。
torch.manual_seed(159)  # 执行当前语句以推进本节示例。
IMAGE_ID159, VOCAB159, HIDDEN159 = 63, 64, 24  # 计算并保存当前步骤的中间状态。
images159 = torch.randn(2, 3, 8, 8)  # 计算并保存当前步骤的中间状态。
ids159 = torch.tensor([[1, IMAGE_ID159, 5, 6, 7], [2, 3, IMAGE_ID159, 8, 9]])  # 计算并保存当前步骤的中间状态。
assert images159.shape == (2, 3, 8, 8)  # 用受控断言验证关键不变量。
assert (ids159 == IMAGE_ID159).sum().item() == 2  # 用受控断言验证关键不变量。
assert HIDDEN159 % 4 == 0  # 用受控断言验证关键不变量。


## 1. Vision encoder 显式 patchify 并保留二维顺序

`8×8` 图像以 `4×4` patch 切成 4 个 token。reshape/permute 的轴顺序必须写清，不能把 channel 与空间像素错排。这里用线性层代替完整 ViT，聚焦跨模态接口。


In [ ]:
class VisionPatchEncoder159(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, channels=3, patch=4, vision_dim=16, max_patches=16):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.patch = patch  # 计算并保存当前步骤的中间状态。
        self.proj = nn.Linear(channels * patch * patch, vision_dim)  # 计算并保存当前步骤的中间状态。
        self.position = nn.Parameter(torch.zeros(1, max_patches, vision_dim))  # 计算并保存当前步骤的中间状态。

    def forward(self, images):  # 定义本节可复用的核心函数。
        # 从 [B,C,H,W] 变为按行扫描的 [B,N,C*P*P] patch 序列。
        b, c, h, w = images.shape  # 计算并保存当前步骤的中间状态。
        if h % self.patch or w % self.patch:  # 按当前条件选择后续控制路径。
            raise ValueError("image size must be divisible by patch")  # 遇到非法合同立即显式失败。
        p = self.patch  # 计算并保存当前步骤的中间状态。
        patches = images.reshape(b, c, h // p, p, w // p, p).permute(0, 2, 4, 1, 3, 5).reshape(b, -1, c * p * p)  # 计算并保存当前步骤的中间状态。
        return self.proj(patches) + self.position[:, : patches.shape[1]]  # 返回当前分支计算出的结果。

vision159 = VisionPatchEncoder159()  # 计算并保存当前步骤的中间状态。
visual_raw159 = vision159(images159)  # 计算并保存当前步骤的中间状态。
assert visual_raw159.shape == (2, 4, 16)  # 用受控断言验证关键不变量。
assert vision159.position.requires_grad  # 用受控断言验证关键不变量。
assert torch.isfinite(visual_raw159).all()  # 用受控断言验证关键不变量。


## 2. Projector 负责维度/分布对齐，不等于视觉理解全部发生在此

两层 MLP 把 `vision_dim` 映射到 LLM hidden。它必须输出与 token embedding 相同 dtype/hidden，且参数版本与视觉/语言 checkpoint 绑定。只训练 projector 计算便宜，但表达能力受冻结两塔限制。


In [ ]:
class MLPProjector159(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vision_dim, language_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.fc1 = nn.Linear(vision_dim, language_dim)  # 计算并保存当前步骤的中间状态。
        self.fc2 = nn.Linear(language_dim, language_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, visual_tokens):  # 定义本节可复用的核心函数。
        # GELU 后再投影，输出轴严格保持 [B,N,H_language]。
        return self.fc2(F.gelu(self.fc1(visual_tokens)))  # 返回当前分支计算出的结果。

projector159 = MLPProjector159(16, HIDDEN159)  # 计算并保存当前步骤的中间状态。
visual159 = projector159(visual_raw159)  # 计算并保存当前步骤的中间状态。
assert visual159.shape == (2, 4, HIDDEN159)  # 用受控断言验证关键不变量。
assert visual159.dtype == visual_raw159.dtype  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in projector159.parameters()) > 0  # 用受控断言验证关键不变量。


## 3. `<image>` 占位符被 N 个视觉 token 替换

每个样本必须恰好找到约定数量的占位符。插入后序列长度从 `T` 变为 `T-1+N`；视觉位置 label 设为 `-100`，其余文本 token 保持因果 LM 目标。


In [ ]:
def splice_visual159(text_embeddings, input_ids, visual_tokens):  # 定义本节可复用的核心函数。
    # 逐样本按占位位置拼接；本教学 batch 的视觉 token 数相同，无需 padding。
    fused, labels, modalities = [], [], []  # 计算并保存当前步骤的中间状态。
    for b in range(len(input_ids)):  # 遍历输入元素以累积或检查结果。
        positions = torch.where(input_ids[b] == IMAGE_ID159)[0]  # 计算并保存当前步骤的中间状态。
        if len(positions) != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("exactly one image placeholder required")  # 遇到非法合同立即显式失败。
        pos = int(positions[0])  # 计算并保存当前步骤的中间状态。
        fused.append(torch.cat([text_embeddings[b, :pos], visual_tokens[b], text_embeddings[b, pos + 1:]], dim=0))  # 计算并保存当前步骤的中间状态。
        labels.append(torch.cat([input_ids[b, :pos], torch.full((visual_tokens.shape[1],), -100, dtype=torch.long), input_ids[b, pos + 1:]]))  # 计算并保存当前步骤的中间状态。
        modalities.append(["text"] * pos + ["image"] * visual_tokens.shape[1] + ["text"] * (input_ids.shape[1] - pos - 1))  # 执行当前语句以推进本节示例。
    return torch.stack(fused), torch.stack(labels), modalities  # 返回当前分支计算出的结果。

embed159 = nn.Embedding(VOCAB159, HIDDEN159)  # 计算并保存当前步骤的中间状态。
text159 = embed159(ids159)  # 计算并保存当前步骤的中间状态。
fused159, labels159, modalities159 = splice_visual159(text159, ids159, visual159)  # 计算并保存当前步骤的中间状态。
assert fused159.shape == (2, 8, HIDDEN159)  # 用受控断言验证关键不变量。
assert (labels159 == -100).sum().item() == 8  # 用受控断言验证关键不变量。
assert all(row.count("image") == 4 for row in modalities159)  # 用受控断言验证关键不变量。


## 4. 手写 causal self-attention 保持视觉前缀可见

图像 token 位于后续文本之前，因此回答 token 可以读取全部视觉 token；视觉 token 不能读取未来回答。这里显式构造下三角 mask，并手写 QKV、多头拆分与 softmax。


In [ ]:
class CausalBlock159(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden, heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.heads, self.dim = heads, hidden // heads  # 计算并保存当前步骤的中间状态。
        self.norm1, self.norm2 = nn.LayerNorm(hidden), nn.LayerNorm(hidden)  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(hidden, 3 * hidden)  # 计算并保存当前步骤的中间状态。
        self.out = nn.Linear(hidden, hidden)  # 计算并保存当前步骤的中间状态。
        self.ff = nn.Sequential(nn.Linear(hidden, 4 * hidden), nn.GELU(), nn.Linear(4 * hidden, hidden))  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        # Pre-norm 后拆成 [B,heads,T,dim]，mask future key 再合并残差。
        b, t, h = x.shape  # 计算并保存当前步骤的中间状态。
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)  # 计算并保存当前步骤的中间状态。
        reshape = lambda z: z.view(b, t, self.heads, self.dim).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        q, k, v = map(reshape, (q, k, v))  # 计算并保存当前步骤的中间状态。
        score = q @ k.transpose(-2, -1) / math.sqrt(self.dim)  # 计算并保存当前步骤的中间状态。
        causal = torch.tril(torch.ones(t, t, dtype=torch.bool, device=x.device))  # 计算并保存当前步骤的中间状态。
        attention = torch.softmax(score.masked_fill(~causal, float("-inf")), dim=-1)  # 计算并保存当前步骤的中间状态。
        merged = (attention @ v).transpose(1, 2).contiguous().view(b, t, h)  # 计算并保存当前步骤的中间状态。
        x = x + self.out(merged)  # 计算并保存当前步骤的中间状态。
        return x + self.ff(self.norm2(x)), attention  # 返回当前分支计算出的结果。

block159 = CausalBlock159(HIDDEN159, heads=4)  # 计算并保存当前步骤的中间状态。
hidden159, attention159 = block159(fused159)  # 计算并保存当前步骤的中间状态。
assert hidden159.shape == fused159.shape  # 用受控断言验证关键不变量。
assert attention159.shape == (2, 4, 8, 8)  # 用受控断言验证关键不变量。
assert torch.all(attention159[..., torch.triu(torch.ones(8, 8), diagonal=1).bool()] == 0)  # 用受控断言验证关键不变量。


## 5. `TinyLLaVA.forward` 串联视觉塔、投影器、LLM 与 loss

Forward 内完成占位替换与 shifted loss。训练时只统计非 `-100` target；推理时返回 logits 和视觉 token 数。输出词表仍是语言词表，视觉 token 只是 hidden 序列元素。


In [ ]:
class TinyLLaVA159(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.vision = VisionPatchEncoder159()  # 计算并保存当前步骤的中间状态。
        self.projector = MLPProjector159(16, HIDDEN159)  # 计算并保存当前步骤的中间状态。
        self.token_embedding = nn.Embedding(VOCAB159, HIDDEN159)  # 计算并保存当前步骤的中间状态。
        self.block = CausalBlock159(HIDDEN159, 4)  # 计算并保存当前步骤的中间状态。
        self.lm_head = nn.Linear(HIDDEN159, VOCAB159, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, images, input_ids):  # 定义本节可复用的核心函数。
        # 占位 token 的临时 embedding 会被视觉序列覆盖，不参与最终 hidden。
        visual = self.projector(self.vision(images))  # 计算并保存当前步骤的中间状态。
        text = self.token_embedding(input_ids)  # 计算并保存当前步骤的中间状态。
        fused, labels, _ = splice_visual159(text, input_ids, visual)  # 计算并保存当前步骤的中间状态。
        hidden, attention = self.block(fused)  # 计算并保存当前步骤的中间状态。
        logits = self.lm_head(hidden)  # 计算并保存当前步骤的中间状态。
        targets = labels[:, 1:]  # 计算并保存当前步骤的中间状态。
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB159), targets.reshape(-1), ignore_index=-100)  # 计算并保存当前步骤的中间状态。
        return {"logits": logits, "loss": loss, "attention": attention, "visual_tokens": visual.shape[1]}  # 返回当前分支计算出的结果。

model159 = TinyLLaVA159()  # 计算并保存当前步骤的中间状态。
result159 = model159(images159, ids159)  # 计算并保存当前步骤的中间状态。
result159["loss"].backward()  # 执行当前语句以推进本节示例。
assert result159["logits"].shape == (2, 8, VOCAB159)  # 用受控断言验证关键不变量。
assert result159["visual_tokens"] == 4  # 用受控断言验证关键不变量。
assert torch.isfinite(result159["loss"]) and model159.projector.fc1.weight.grad is not None  # 用受控断言验证关键不变量。


## 6. 两阶段训练通过 `requires_grad` 明确冻结合同

Stage 1 常只训练 projector；Stage 2 可解冻 projector 与部分/全部 LLM，视觉塔是否解冻取决于数据和预算。优化器必须在冻结后创建，否则可能保留不再更新参数的状态。


In [ ]:
def configure_stage159(model, stage):  # 定义本节可复用的核心函数。
    # 先全冻结，再按阶段白名单打开模块，避免遗漏旧 requires_grad 状态。
    for parameter in model.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.requires_grad = False  # 计算并保存当前步骤的中间状态。
    if stage == "align-projector":  # 按当前条件选择后续控制路径。
        for parameter in model.projector.parameters(): parameter.requires_grad = True  # 遍历输入元素以累积或检查结果。
    elif stage == "instruction-tune":  # 按当前条件选择后续控制路径。
        for module in (model.projector, model.block, model.lm_head):  # 遍历输入元素以累积或检查结果。
            for parameter in module.parameters(): parameter.requires_grad = True  # 遍历输入元素以累积或检查结果。
    else:  # 处理前置条件不成立的分支。
        raise ValueError("unknown stage")  # 遇到非法合同立即显式失败。
    return {name for name, parameter in model.named_parameters() if parameter.requires_grad}  # 返回当前分支计算出的结果。

stage1_159 = configure_stage159(model159, "align-projector")  # 计算并保存当前步骤的中间状态。
stage2_159 = configure_stage159(model159, "instruction-tune")  # 计算并保存当前步骤的中间状态。
assert stage1_159 and all(name.startswith("projector.") for name in stage1_159)  # 用受控断言验证关键不变量。
assert len(stage2_159) > len(stage1_159)  # 用受控断言验证关键不变量。
assert not any(name.startswith("vision.") for name in stage2_159)  # 用受控断言验证关键不变量。


## 7. Any-resolution 策略必须先预算视觉 token

高分辨率图像切更多 tile 会提高局部细节，也以 token 数增加 LLM attention 成本。下面按 tile 网格估算 patch token，并在超过预算时拒绝，而不是静默缩小导致坐标语义变化。


In [ ]:
def visual_token_budget159(height, width, tile_size=336, patch_size=14, max_tokens=4096):  # 定义本节可复用的核心函数。
    # 每个 tile 产生固定 patch 网格，边缘 tile 仍 padding 到完整大小。
    rows, cols = math.ceil(height / tile_size), math.ceil(width / tile_size)  # 计算并保存当前步骤的中间状态。
    per_tile = (tile_size // patch_size) ** 2  # 计算并保存当前步骤的中间状态。
    tokens = rows * cols * per_tile  # 计算并保存当前步骤的中间状态。
    return {"rows": rows, "cols": cols, "tokens": tokens, "admit": tokens <= max_tokens}  # 返回当前分支计算出的结果。

small_budget159 = visual_token_budget159(336, 336)  # 计算并保存当前步骤的中间状态。
wide_budget159 = visual_token_budget159(672, 1344)  # 计算并保存当前步骤的中间状态。
assert small_budget159["tokens"] == 576  # 用受控断言验证关键不变量。
assert wide_budget159["rows"] == 2 and wide_budget159["cols"] == 4  # 用受控断言验证关键不变量。
assert not wide_budget159["admit"] and small_budget159["admit"]  # 用受控断言验证关键不变量。


## 8. 多模态验收区分语言正确、视觉 grounding 与幻觉

仅看回答 fluency 会漏掉“图中不存在对象”。受控 grader 从 claim 集合比较可见对象，分别统计 grounded precision、coverage 和 unsupported claims；真实系统还要验证 box/region、OCR 与拒答。


In [ ]:
def grounding_metrics159(visible_objects, claimed_objects, required_objects):  # 定义本节可复用的核心函数。
    # 集合 oracle 只验证对象存在性，不能替代细粒度空间关系评测。
    visible, claimed, required = map(set, (visible_objects, claimed_objects, required_objects))  # 计算并保存当前步骤的中间状态。
    supported = claimed & visible  # 计算并保存当前步骤的中间状态。
    precision = len(supported) / max(len(claimed), 1)  # 计算并保存当前步骤的中间状态。
    coverage = len(claimed & required) / max(len(required), 1)  # 计算并保存当前步骤的中间状态。
    return {"precision": precision, "coverage": coverage, "unsupported": sorted(claimed - visible)}  # 返回当前分支计算出的结果。

grounded159 = grounding_metrics159(["cat", "sofa"], ["cat", "sofa"], ["cat"])  # 计算并保存当前步骤的中间状态。
hallucinated159 = grounding_metrics159(["cat", "sofa"], ["cat", "car"], ["cat"])  # 计算并保存当前步骤的中间状态。
assert grounded159["precision"] == 1.0  # 用受控断言验证关键不变量。
assert hallucinated159["unsupported"] == ["car"]  # 用受控断言验证关键不变量。
assert hallucinated159["coverage"] == 1.0 and hallucinated159["precision"] < 1.0  # 用受控断言验证关键不变量。


## 面试总结

- 图像先变 patch token，projector 对齐到 LLM hidden，再替换文本中的 `<image>` 占位符。
- 融合序列进入 causal decoder；视觉位置 loss mask 为 ignore，回答 token 才提供语言监督。
- 两阶段训练先对齐 projector，再做指令微调；冻结集合和三份 checkpoint revision 都是接口合同。
- 高分辨率提升细节但增加视觉 token 与二次 attention 成本，评测必须单独看 grounding/幻觉。

延伸阅读：[Visual Instruction Tuning / LLaVA](https://arxiv.org/abs/2304.08485)、[LLaVA 1.5](https://arxiv.org/abs/2310.03744)、[Flamingo](https://arxiv.org/abs/2204.14198)。
